# AEU-OMR: nota resminden nota dosyasına

Bu defter, basılı ya da fotoğrafı çekilmiş bir Türk müziği notasını okur ve **MusicXML** nota dosyasına çevirir.

**Ne yapar?**
1. Sayfadaki porteleri bulur.
2. Her porteyi AEU-OMR modeli okur: notalar, süreler, koma değerli arıza işaretleri, donanım, usûl rakamları.
3. Ayrı bir adımda, sayfanın başlığındaki yazıları (eser adı, makam, form, usûl, bestekâr, güfte) metin okuma (OCR) ile okur.
4. Hepsini nota programlarının açabildiği bir MusicXML dosyasına yazar. Her notanın çalınacağı perde, donanım ve ölçü içi kurallarıyla bulunur.

**Nasıl kullanılır?**
1. *Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU* seçin.
2. Drive'da `homr_makam/demo/girdi` klasörüne nota fotoğraflarını (JPG, PNG) ya da PDF'leri koyun.
3. Hücreleri sırayla çalıştırın. Okunan nota dosyaları `homr_makam/demo/cikti` klasörüne yazılır.

In [ ]:
#@title 1. Kurulum (bir kez; birkaç dakika sürer)
MODEL = 'v24 (en yeni)'  #@param ["v24 (en yeni)", "v20 (tezdeki)"]

from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess
DRIVE = '/content/drive/MyDrive/homr_makam'
KOD = '/content/aeu-omr'
if not os.path.isdir(KOD):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'aeu-omr',
                    'https://github.com/mtalhabalci/aeu-omr.git', KOD], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'musicxml==1.4', 'x-transformers', 'timm', 'einops',
                'albumentations', 'rapidocr==3.9.2', 'onnxruntime', 'pymupdf', 'fastapi', 'python-multipart',
                'verovio'], check=True)
os.chdir(KOD)
sys.path.insert(0, KOD)

# modelin ağırlıkları Drive'dan
AGIRLIK = {'v24 (en yeni)': 'pytorch_model_1-819bee2ae062dff09fd97af4dfae98dd1709ceb3.pth',
           'v20 (tezdeki)': 'pytorch_model_1-111a84bed30c8c6a373ad135e05633ccd8c63b11.pth'}[MODEL]
hedef = f'{KOD}/training/architecture/transformer/{AGIRLIK}'
if not os.path.exists(hedef):
    shutil.copy(f'{DRIVE}/{AGIRLIK}', hedef)
# yazılı tempoyu (♩=112 gibi) okuyan küçük modeller
os.makedirs(f'{KOD}/training/architecture/tempo', exist_ok=True)
for k in ('1', '2', '3', 'aralik'):
    kaynak = f'{DRIVE}/tempo/tempo_okuyucu_{k}.pt'
    if os.path.exists(kaynak):
        shutil.copy(kaynak, f'{KOD}/training/architecture/tempo/')
# sayfadaki porteleri bulan ağ, Drive'dan
SEGNET = 'segnet_308-3296ccd40960f90ca6ab9c035cca945675d30a0f.onnx'
hedef_segnet = os.path.join(KOD, 'homr', 'segmentation', SEGNET)
if not os.path.exists(hedef_segnet):
    shutil.copy(f'{DRIVE}/{SEGNET}', hedef_segnet)

os.environ['OMR_MODEL'] = hedef
os.environ['OMR_MODEL_SURUM'] = MODEL
os.environ['OMR_YAZILIM'] = 'AEU-OMR'   # MusicXML dosyasındaki yazılım adı
from servis.nota_servisi import Reader
okuyucu = Reader()
print('Hazır:', MODEL, '| cihaz:', okuyucu.device)

## 2. Notaları oku

Her dosya için üç şey gösterilir:

- **Başlıktan okunan:** sayfanın başındaki yazılardan çıkarılan eser adı, makam, form, usûl, bestekâr ve güfte (OCR).
- **Modelin yazdığı nota:** modelin okuduğu nota, nota programına verilmiş ve yeniden dizilmiş hâliyle.
- **Nota nota tablo:** Her nota için ölçüsü; Batı adı ve kaç koma tiz ya da pes çalındığı; Türk müziğindeki perde adı
  (Hatiboğlu, 2015, s. 13); sayfada önünde çizili arıza işareti (işaretsiz notalar donanımdan ya da ölçüdeki önceki
  işaretten perdesini alır); süresi.

In [ ]:
#@title 2. Drive'daki notaları oku ve göster
GIRDI = '/content/drive/MyDrive/homr_makam/demo/girdi'  #@param {type:"string"}
CIKTI = '/content/drive/MyDrive/homr_makam/demo/cikti'  #@param {type:"string"}
TABLODA_NOTA = 60  #@param {type:"integer"}

import json, html as _html, xml.etree.ElementTree as ET
import verovio
from IPython.display import SVG, HTML, display

PERDE = {"D4+0": "Yegâh", "D4+4": "Kaba Nim Hisâr", "E4-5": "Kaba Nim Hisâr", "D4+5": "Kaba Hisâr", "E4-4": "Kaba Hisâr", "D4+8": "Kaba Dik Hisâr", "E4-1": "Kaba Dik Hisâr", "E4+0": "Hüseyni Aşîrân", "F4+0": "Acem Aşîrân", "F4+1": "Dik Acem Aşîrân", "G4-8": "Dik Acem Aşîrân", "F4+4": "Irâk", "G4-5": "Irâk", "F4+5": "Geveşt", "G4-4": "Geveşt", "F4+8": "Dik Geveşt", "G4-1": "Dik Geveşt", "G4+0": "Rast", "G4+4": "Nim Zengüle", "A4-5": "Nim Zengüle", "G4+5": "Zengüle", "A4-4": "Zengüle", "G4+8": "Dik Zengüle", "A4-1": "Dik Zengüle", "A4+0": "Dügâh", "A4+4": "Kürdî", "B4-5": "Kürdî", "A4+5": "Dik Kürdî", "B4-4": "Dik Kürdî", "A4+8": "Segâh", "B4-1": "Segâh", "B4+0": "Bûselik", "C5-1": "Dik Bûselik", "C5+0": "Çargâh", "C5+4": "Nim Hicâz", "D5-5": "Nim Hicâz", "C5+5": "Hicâz", "D5-4": "Hicâz", "C5+8": "Dik Hicâz", "D5-1": "Dik Hicâz", "D5+0": "Nevâ", "D5+4": "Nim Hisâr", "E5-5": "Nim Hisâr", "D5+5": "Hisâr", "E5-4": "Hisâr", "D5+8": "Dik Hisâr", "E5-1": "Dik Hisâr", "E5+0": "Hüseynî", "F5+0": "Acem", "F5+1": "Dik Acem", "G5-8": "Dik Acem", "F5+4": "Evc", "G5-5": "Evc", "F5+5": "Mâhûr", "G5-4": "Mâhûr", "F5+8": "Dik Mâhûr", "G5-1": "Dik Mâhûr", "G5+0": "Gerdâniye", "G5+4": "Nim Şehnâz", "A5-5": "Nim Şehnâz", "G5+5": "Şehnâz", "A5-4": "Şehnâz", "G5+8": "Dik Şehnâz", "A5-1": "Dik Şehnâz", "A5+0": "Muhayyer", "A5+4": "Sünbüle", "B5-5": "Sünbüle", "A5+5": "Dik Sünbüle", "B5-4": "Dik Sünbüle", "A5+8": "Tiz Segâh", "B5-1": "Tiz Segâh", "B5+0": "Tiz Bûselik", "C6-1": "Tiz Dik Bûselik", "C6+0": "Tiz Çargâh", "C6+4": "Tiz Nim Hicâz", "D6-5": "Tiz Nim Hicâz", "C6+5": "Tiz Hicâz", "D6-4": "Tiz Hicâz", "C6+8": "Tiz Dik Hicâz", "D6-1": "Tiz Dik Hicâz", "D6+0": "Tiz Nevâ"}
HARF = {'C': 'Do', 'D': 'Re', 'E': 'Mi', 'F': 'Fa', 'G': 'Sol', 'A': 'La', 'B': 'Si'}
SURE = {'breve': 'çift birlik', 'whole': 'birlik', 'half': 'ikilik', 'quarter': 'dörtlük', 'eighth': 'sekizlik',
        '16th': 'onaltılık', '32nd': 'otuzikilik', '64th': 'altmışdörtlük'}
ISARET = {'accidentalKomaSharp': 'koma diyezi (1)', 'accidentalBakiyeSharp': 'bakiyye diyezi (4)',
          'accidentalKucukMucennebSharp': 'küçük mücenneb diyezi (5)',
          'accidentalBuyukMucennebSharp': 'büyük mücenneb diyezi (8)', 'accidentalKomaFlat': 'koma bemolü (1)',
          'accidentalBakiyeFlat': 'bakiyye bemolü (4)', 'accidentalKucukMucennebFlat': 'küçük mücenneb bemolü (5)',
          'accidentalBuyukMucennebFlat': 'büyük mücenneb bemolü (8)', 'accidentalNatural': 'natürel',
          'accidentalDoubleSharp': 'çift diyez (9)', 'accidentalDoubleFlat': 'çift bemol (9)',
          'accidental2CommaSharp': '2 koma diyez', 'accidental3CommaSharp': '3 koma diyez',
          'accidental2CommaFlat': '2 koma bemol', 'accidental3CommaFlat': '3 koma bemol'}
ALANLAR = [('eserAdi', 'Eser'), ('makam', 'Makam'), ('form', 'Form'), ('usul', 'Usûl'), ('bestekar', 'Bestekâr'),
           ('gufte', 'Güfte'), ('yore', 'Yöre'), ('kaynak', 'Kaynak')]


def perde_adi(step, octave, koma):
    """Perde adı; yegâh ile tiz nevâ arasının dışında kalanlara 'Kaba' / 'Tiz' eklenir."""
    for fark, on_ek in ((0, ''), (1, 'Kaba '), (-1, 'Tiz ')):
        ad = PERDE.get(f'{step}{octave + fark}{koma:+d}')
        if ad:
            return on_ek + ad
    return ''


def tablo(xml, en_cok):
    satirlar = []
    for olcu in ET.fromstring(xml.encode('utf-8')).iter('measure'):
        for nota in olcu.findall('note'):
            sure = SURE.get(nota.findtext('type'), nota.findtext('type') or '') + '.' * len(nota.findall('dot'))
            if nota.find('rest') is not None:
                satirlar.append((olcu.get('number'), 'sus', '', '', sure))
            elif nota.find('pitch') is not None:
                p = nota.find('pitch')
                step, octave = p.findtext('step'), int(p.findtext('octave'))
                koma = round(float(p.findtext('alter') or 0) * 9 / 2)   # MusicXML: yarım ses; 1 koma = 2/9 yarım ses
                isaret = nota.find('accidental')
                isaret = ISARET.get(isaret.get('smufl'), isaret.text) if isaret is not None else ''
                adi = f'{HARF[step]}{octave}' + (f' ({koma:+d} koma)' if koma else '')
                satirlar.append((olcu.get('number'), adi, perde_adi(step, octave, koma), isaret, sure))
            if len(satirlar) >= en_cok:
                break
        if len(satirlar) >= en_cok:
            break
    hucre = lambda x: f'<td style="padding:2px 10px">{_html.escape(str(x))}</td>'
    bas = ''.join(f'<th style="padding:2px 10px;text-align:left">{b}</th>'
                  for b in ('Ölçü', 'Nota', 'Perde', 'Sayfadaki işaret', 'Süre'))
    return '<table><tr>' + bas + '</tr>' + ''.join('<tr>' + ''.join(hucre(x) for x in s) + '</tr>'
                                                   for s in satirlar) + '</table>'


os.makedirs(GIRDI, exist_ok=True)
os.makedirs(CIKTI, exist_ok=True)
dosyalar = sorted(f for f in os.listdir(GIRDI) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.pdf')))
if not dosyalar:
    print(f'{GIRDI} boş: oraya nota fotoğrafı (JPG, PNG) ya da PDF koyun.')
for ad in dosyalar:
    display(HTML(f'<h2>{_html.escape(ad)}</h2>'))
    try:
        sonuc = okuyucu.read(os.path.join(GIRDI, ad))
    except Exception as hata:  # noqa: BLE001
        print('Okunamadı:', hata)
        continue
    xml = sonuc['musicxml']
    kok = os.path.splitext(ad)[0]
    with open(os.path.join(CIKTI, kok + '.musicxml'), 'w', encoding='utf-8') as f:
        f.write(xml)
    baslik = sonuc.get('baslikBilgisi') or {}
    okunan = [f'<b>{e}:</b> {_html.escape(str(baslik[k]))}' for k, e in ALANLAR if baslik.get(k)]
    display(HTML('<p><b>Başlıktan okunan (OCR):</b> ' + (' · '.join(okunan) if okunan else '<i>okunamadı</i>')
                 + '</p>'))
    print(f"Okunan porte: {len(sonuc.get('porteler') or [])} | uyarılar: {sonuc.get('uyarilar') or 'yok'}")
    display(HTML('<p><b>Modelin yazdığı nota:</b></p>'))
    tk = verovio.toolkit()
    tk.setOptions({'pageWidth': 2100, 'adjustPageHeight': True, 'scale': 35, 'header': 'none', 'footer': 'none'})
    tk.loadData(xml)
    for sayfa in range(1, tk.getPageCount() + 1):
        display(SVG(tk.renderToSVG(sayfa)))
    display(HTML(f'<p><b>Nota nota (ilk {TABLODA_NOTA}):</b></p>' + tablo(xml, TABLODA_NOTA)))
    print('MusicXML:', os.path.join(CIKTI, kok + '.musicxml'))